In [ ]:
# ── IdiomBERT Exp 02 (XLM-R QA-vs-BIO) — multi-seed canonical launcher ────
# FIRST: Runtime → Change runtime type → T4 GPU.
# Seed 42 already canonical in pipeline_eval_results.json.
# Run this notebook twice more with SEED = '123' and SEED = '7'.
import os, subprocess
from pathlib import Path

SEED = '42'   # <-- change to '123' or '7' for additional seeds

# 1. Fail loud if no GPU (fixed args, no injection risk)
assert subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0, \
    'No GPU — set Runtime → T4 GPU'

# 2. Clone or pull latest — MUST push commits before running here
REPO = '/content/Idiomator_Research'
if Path(REPO).exists():
    !cd $REPO && git pull --ff-only
else:
    !git clone https://github.com/JustLetMeBeHello/Idiomator_Research.git $REPO
%cd $REPO
!pip install -q -r Requirements.txt

# 3. Mount Drive — durable output dir
from google.colab import drive
drive.mount('/content/drive')
DRIVE_OUT = '/content/drive/MyDrive/IdiomatorRigor'
os.environ['DRIVE_OUT'] = DRIVE_OUT
os.environ['SEED'] = SEED
Path(DRIVE_OUT).mkdir(parents=True, exist_ok=True)

# 4. Train (script: rm -rf models/rigor_xlmr_joint_s{SEED} then ln -s Drive, gates before training)
LOG = f"{DRIVE_OUT}/exp02_seed{SEED}_console.log"
!bash experiments/rigor/run_02_xlmr_qa_vs_bio.sh 2>&1 | tee -a "$LOG"
print(f'\nDone. Log at {LOG}')


In [ ]:
# ── Register results in pipeline_eval_results.json (run after training) ──────
# Writes rigor_xlmr_joint_s{SEED} and rigor_bio_xlmr_s{SEED} to canonical json.
import os
SEED     = os.environ.get('SEED', '42')
DRIVE_OUT = os.environ.get('DRIVE_OUT', '/content/drive/MyDrive/IdiomatorRigor')

JOINT_DIR = f'models/rigor_xlmr_joint_s{SEED}'
BIO_DIR   = f'models/rigor_bio_xlmr_s{SEED}'

# Verify Drive-backed symlinks survived (persistence trap check)
for d in [JOINT_DIR, BIO_DIR]:
    assert os.path.islink(d), f'{d} not a symlink — training output ephemeral or not complete'
    assert os.path.exists(d), f'{d} symlink target missing on Drive'
    print(f'✓ {d} -> {os.path.realpath(d)}')

# --xlmr_seed controls both pred paths (default models/rigor_xlmr_joint_s{seed}/...)
# and output keys (rigor_xlmr_joint_s{seed}) so seeds don't overwrite each other
!python Evaluation/Full_evaluation.py --xlmr_seed "$SEED" 2>&1 | tail -40


In [ ]:
# ── Gate: QA vs BIO span exact match, paired bootstrap CI ────────────────────
# Shows raw (artifact-contaminated) gap for sanity check.
# The cleaned-up claim is in run_08_extended_gold (strip normalization).
import json, os, random
from collections import defaultdict

SEED     = os.environ.get('SEED', '42')
DRIVE_OUT = os.environ.get('DRIVE_OUT', '/content/drive/MyDrive/IdiomatorRigor')
QA  = f'{DRIVE_OUT}/rigor_xlmr_joint_s{SEED}/test_predictions.jsonl'
BIO = f'{DRIVE_OUT}/rigor_bio_xlmr_s{SEED}/test_predictions.jsonl'

load = lambda p: [json.loads(l) for l in open(p, encoding='utf-8')]

def rows(path):
    out = {}
    for r in load(path):
        gs, ge = r.get('span_start'), r.get('span_end')
        ps, pe = r.get('pred_span_start'), r.get('pred_span_end')
        ex = int(ps is not None and gs is not None and ps == gs and pe == ge)
        out[r['sentence']] = (r['language'], ex, float(r.get('span_overlap_f1') or 0))
    return out

qa, bio = rows(QA), rows(BIO)
shared = sorted(set(qa) & set(bio))
print(f'QA n={len(qa)}  BIO n={len(bio)}  shared(paired)={len(shared)}')

bylang = defaultdict(list)
for s in shared:
    bylang[qa[s][0]].append((qa[s][1], bio[s][1], qa[s][2], bio[s][2]))

def boot(pairs, n=10000, seed=42):
    rnd = random.Random(seed); N = len(pairs)
    qm = sum(p[0] for p in pairs)/N; bm = sum(p[1] for p in pairs)/N
    g = []
    for _ in range(n):
        s = [pairs[rnd.randrange(N)] for _ in range(N)]
        g.append(sum(a-b for a,b,_,_ in s)/N)
    g.sort()
    return qm, bm, g[int(.025*n)], g[int(.975*n)]

order = ['English','Spanish','Hindi','Telugu','Indonesian']
ALL   = [t for l in order for t in bylang.get(l, [])]
print(f"\n{'lang':11}{'n':>4}{'QA ex':>7}{'BIO ex':>8}{'gap':>7}{'95% CI gap':>16}{'sig':>5} | {'QA ov':>6}{'BIO ov':>7}")
for l in order + ['ALL']:
    pairs = ALL if l == 'ALL' else bylang.get(l)
    if not pairs: continue
    qm, bm, lo, hi = boot(pairs)
    qov = sum(p[2] for p in pairs)/len(pairs)
    bov = sum(p[3] for p in pairs)/len(pairs)
    sig = 'YES' if (lo > 0 or hi < 0) else 'ns'
    print(f'{l:11}{len(pairs):>4}{qm:>7.2f}{bm:>8.2f}{qm-bm:>+7.2f}  [{lo:+.2f},{hi:+.2f}]{sig:>5} | {qov:>6.2f}{bov:>7.2f}')

print('\n(Raw gap = artifact-contaminated. Strip-normalized gap in run_08 = SP family +0.025, at WP floor +0.030.)')